# Week 7 - Advanced Analysis and Tool Development

## Objective

Quantify the effects of VIX and other new features using Week 6 SHAP outputs, test the required 50% volatility spike and 2% rate hike, evaluate both ML pricing routes under counterfactual conditions, and establish a reproducible Streamlit/data-update prototype for Week 8.

## 1. Load Week 2, Week 5, and Week 6 Deliverables

In [1]:
from pathlib import Path
import json
import sys

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def locate_project_root():
    candidates = [Path.cwd(), *Path.cwd().parents, Path(r"G:\JPM-Chooser Option Pricing")]
    for candidate in candidates:
        required = [
            candidate / "Week2" / "processed_data" / "market_data_processed.csv",
            candidate / "Week 4" / "bsm_chooser.py",
            candidate / "Week 5" / "model_results" / "ml_dataset.csv",
            candidate / "Week 6" / "trained_models" / "model_bundle_metadata.json",
        ]
        if all(path.exists() for path in required):
            return candidate
    raise FileNotFoundError("Could not locate the complete Week 2/4/5/6 project chain.")


PROJECT_ROOT = locate_project_root()
WEEK_DIR = Path.cwd()
RESULTS_DIR = WEEK_DIR / "model_results"
FIGURES_DIR = WEEK_DIR / "figures"
LIVE_DIR = WEEK_DIR / "live_data"
for directory in [RESULTS_DIR, FIGURES_DIR, LIVE_DIR]:
    directory.mkdir(exist_ok=True)

sys.path.insert(0, str(WEEK_DIR))
sys.path.insert(0, str(PROJECT_ROOT / "Week 4"))
sys.path.insert(0, str(PROJECT_ROOT / "Week 5"))
from bsm_chooser import simple_chooser_price
from ml_features import PRICING_FEATURES, VOLATILITY_FEATURES
from sensitivity_analysis import apply_ml_scenario, stress_scenarios
from data_update import update_snapshot

with open(WEEK_DIR / "tool_config.json", encoding="utf-8") as stream:
    CONFIG = json.load(stream)
metadata = json.loads((PROJECT_ROOT / "Week 6" / "trained_models" / "model_bundle_metadata.json").read_text(encoding="utf-8"))
volatility_model = joblib.load(PROJECT_ROOT / "Week 6" / "trained_models" / "best_volatility_model.joblib")
pricing_model = joblib.load(PROJECT_ROOT / "Week 6" / "trained_models" / "best_direct_pricing_model.joblib")
dataset = pd.read_csv(PROJECT_ROOT / "Week 5" / "model_results" / "ml_dataset.csv", parse_dates=["Date"])
latest = dataset.iloc[-1].copy()
print(f"Latest analysis row: {latest['Date'].date()}")

Latest analysis row: 2024-11-29


## 2. SHAP-Based Impact of VIX and Related Features

In [2]:
shap_table = pd.read_csv(PROJECT_ROOT / "Week 6" / "model_results" / "shap_feature_importance.csv")
vix_features = {"VIX_Close", "VIX_Return", "VIX_JPM_Correlation_20D"}
shap_table["Feature_Group"] = np.where(shap_table["Feature"].isin(vix_features), "VIX-related", "Other")
grouped = shap_table.groupby(["Approach", "Feature_Group"], as_index=False)["Mean_Absolute_SHAP"].sum()
totals = grouped.groupby("Approach")["Mean_Absolute_SHAP"].transform("sum")
grouped["Share_of_Total_SHAP"] = grouped["Mean_Absolute_SHAP"] / totals

sentiment_gap = pd.DataFrame([{
    "Requested_Feature": "News sentiment score",
    "Available_in_Week2": False,
    "Treatment": "Not fabricated; VIX is analyzed as a market fear/volatility indicator, not relabeled as news sentiment.",
}])
grouped

                       Approach  ... Share_of_Total_SHAP
0      Approach 1 Random Forest  ...            0.592017
1      Approach 1 Random Forest  ...            0.407983
2  Approach 2 Linear Regression  ...            0.949359
3  Approach 2 Linear Regression  ...            0.050641

[4 rows x 4 columns]

## 3. Structural BSM Stress Tests

In [3]:
contract = CONFIG["contract"]
structural_scenarios = stress_scenarios(
    latest, simple_chooser_price,
    strike=contract["strike"], dividend_yield=contract["dividend_yield"],
    choice_time=contract["choice_time_years"], maturity=contract["maturity_years"],
)
structural_scenarios

               Scenario        Spot  ...  Price_Change  Price_Change_Percent
0                  Base  241.119385  ...      0.000000              0.000000
1  50% Volatility Spike  241.119385  ...     16.406649              0.165287
2          2% Rate Hike  241.119385  ...      2.069991              0.020854
3        Combined Shock  241.119385  ...     17.656054              0.177874

[4 rows x 7 columns]

## 4. ML Counterfactual Scenario Testing

In [4]:
scenario_names = ["Base", "50% Volatility Spike", "2% Rate Hike", "Combined Shock"]
ml_scenarios = pd.DataFrame([
    apply_ml_scenario(
        latest, volatility_model, pricing_model,
        VOLATILITY_FEATURES, PRICING_FEATURES,
        simple_chooser_price, name,
        strike=contract["strike"], dividend_yield=contract["dividend_yield"],
        choice_time=contract["choice_time_years"], maturity=contract["maturity_years"],
    )
    for name in scenario_names
])
for column in ["BSM_Price", "Approach1_Price", "Approach2_Price"]:
    ml_scenarios[column + "_Change"] = ml_scenarios[column] - ml_scenarios.loc[0, column]
ml_scenarios

               Scenario  ...  Approach2_Price_Change
0                  Base  ...                     0.0
1  50% Volatility Spike  ...                     0.0
2          2% Rate Hike  ...                     0.0
3        Combined Shock  ...                     0.0

[4 rows x 11 columns]

## 5. VIX Counterfactual Curves

In [5]:
vix_values = np.linspace(CONFIG["vix_grid"]["minimum"], CONFIG["vix_grid"]["maximum"], CONFIG["vix_grid"]["points"])
vix_rows = []
for value in vix_values:
    row = latest.copy()
    row["VIX_Close"] = value
    predicted_vol = float(np.clip(volatility_model.predict(pd.DataFrame([row[VOLATILITY_FEATURES].to_dict()]))[0], 0.01, 2.0))
    approach1 = float(simple_chooser_price(row["Close"], contract["strike"], row["Treasury_Rate_Decimal"], contract["dividend_yield"], predicted_vol, contract["choice_time_years"], contract["maturity_years"]))
    approach2 = max(0.0, float(pricing_model.predict(pd.DataFrame([row[PRICING_FEATURES].to_dict()]))[0]))
    vix_rows.append({"VIX_Close": value, "Predicted_Forward_Volatility": predicted_vol, "Approach1_Price": approach1, "Approach2_Price": approach2})
vix_counterfactual = pd.DataFrame(vix_rows)
vix_counterfactual.head()

   VIX_Close  Predicted_Forward_Volatility  Approach1_Price  Approach2_Price
0       10.0                      0.120153        91.707111              0.0
1       10.5                      0.120153        91.707111              0.0
2       11.0                      0.120153        91.707111              0.0
3       11.5                      0.120153        91.707111              0.0
4       12.0                      0.120153        91.707111              0.0

## 6. Reproducible Data-Update Module Check

In [6]:
# Offline mode guarantees reproducibility inside the submitted notebook.
# The Streamlit button and command-line --online path exercise the online updater.
snapshot = update_snapshot(PROJECT_ROOT, LIVE_DIR, online=False)
snapshot

{'as_of_date': '2024-12-30', 'retrieved_at_utc': '2026-09-07T09:57:43.434790+00:00', 'mode': 'offline_week2_fallback', 'jpm_close': 231.0775604248047, 'vix_close': 17.399999618530273, 'risk_free_rate': 0.0455, 'historical_volatility_20d': 0.1906247800741619, 'sources': {'JPM_and_VIX': 'G:\\JPM-Chooser Option Pricing\\Week2\\processed_data\\market_data_processed.csv', 'risk_free_rate': 'G:\\JPM-Chooser Option Pricing\\Week2\\processed_data\\market_data_processed.csv'}, 'update_status': 'offline_requested', 'online_error': None}

## 7. Save Week 7 Results and Figures

In [7]:
grouped.to_csv(RESULTS_DIR / "shap_feature_group_impact.csv", index=False)
sentiment_gap.to_csv(RESULTS_DIR / "sentiment_data_gap.csv", index=False)
structural_scenarios.to_csv(RESULTS_DIR / "structural_stress_scenarios.csv", index=False)
ml_scenarios.to_csv(RESULTS_DIR / "ml_counterfactual_scenarios.csv", index=False)
vix_counterfactual.to_csv(RESULTS_DIR / "vix_counterfactual_curve.csv", index=False)

plt.style.use("seaborn-v0_8-whitegrid")
fig, ax = plt.subplots(figsize=(9, 5))
plot_data = structural_scenarios.set_index("Scenario")
ax.bar(plot_data.index, plot_data["Chooser_BSM_Price"], color=["#5b9bd5", "#ed7d31", "#70ad47", "#c00000"])
ax.set(title="Chooser BSM Price under Required Stress Scenarios", ylabel="Chooser price ($)", xlabel="")
ax.tick_params(axis="x", rotation=15)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "structural_stress_scenarios.png", dpi=180)
plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.7))
axes[0].plot(vix_counterfactual["VIX_Close"], vix_counterfactual["Predicted_Forward_Volatility"], color="#ed7d31")
axes[0].set(title="VIX Impact on Predicted Volatility", xlabel="VIX", ylabel="Predicted forward volatility")
axes[1].plot(vix_counterfactual["VIX_Close"], vix_counterfactual["Approach1_Price"], label="Approach 1")
axes[1].plot(vix_counterfactual["VIX_Close"], vix_counterfactual["Approach2_Price"], label="Approach 2")
axes[1].set(title="VIX Impact on ML Pricing", xlabel="VIX", ylabel="Price ($)")
axes[1].legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "vix_counterfactual_impact.png", dpi=180)
plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 5))
width = 0.25
x = np.arange(len(ml_scenarios))
for offset, column, label in [(-width, "BSM_Price", "BSM"), (0, "Approach1_Price", "Approach 1"), (width, "Approach2_Price", "Approach 2")]:
    ax.bar(x + offset, ml_scenarios[column], width, label=label)
ax.set_xticks(x, ml_scenarios["Scenario"], rotation=15)
ax.set(title="BSM and ML Prices under Counterfactual Scenarios", ylabel="Price ($)")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "ml_scenario_comparison.png", dpi=180)
plt.close(fig)

print(f"Saved {len(list(RESULTS_DIR.glob('*.csv')))} tables, {len(list(FIGURES_DIR.glob('*.png')))} figures, and an offline data snapshot.")

Saved 5 tables, 3 figures, and an offline data snapshot.


# Week 7 Conclusion

The advanced analysis remains connected to the Week 2 market features and the Week 3/4 chooser engine. VIX-related SHAP contributions are reported separately from news sentiment; because Week 2 did not collect a news-sentiment series, no synthetic sentiment score is invented. The required volatility and rate shocks are evaluated for BSM and both ML routes. The Streamlit prototype and data updater provide the application structure for Week 8, with explicit online/fallback status.